In [ ]:
# imports
import kagglehub
from kagglehub import KaggleDatasetAdapter
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, ConcatDataset
import os
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler

In [ ]:
# download dataset

local_data_path = Path("../data")
dataset_base_path = Path(kagglehub.dataset_download("crawford/cat-dataset"))
dataset_base_path = dataset_base_path / "cats"
oreo_path = local_data_path / "oreo"
not_oreo_path = local_data_path / "not_oreo"

oreo_path.mkdir(exist_ok=True)
not_oreo_path.mkdir(exist_ok=True)

for image in local_data_path.glob("*.jpg"):
    shutil.move(str(image), str(oreo_path / image.name))

for subdir in dataset_base_path.iterdir():
    if subdir.is_dir():
        for image in subdir.glob("*.jpg"):
            new_name = f"{subdir.name}_{image.name}"
            shutil.move(str(image), str(not_oreo_path / new_name))

In [ ]:
print(oreo_path)
print(not_oreo_path)

In [ ]:
# preprocessing

# Resize, Augment, and Normalize RGB
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2), # random brightness
    transforms.ToTensor()
])

full_dataset = datasets.ImageFolder(root=local_data_path, transform=transform)
print(type(full_dataset))
print(full_dataset)
print(f"classes found: {full_dataset.classes}")
print(f"mapping: {full_dataset.class_to_idx}")

train_loader = DataLoader(full_dataset, batch_size=32, shuffle=True)
print(type(train_loader))
print(train_loader)

In [ ]:
# Split

targets = np.array(full_dataset.targets)
indices = np.arange(len(full_dataset))

# 80% Train
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=targets,
    random_state=42
)

# 10% validation
# 10% test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=targets[temp_idx],
    random_state=42
)

train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)
test_dataset = Subset(full_dataset, test_idx)

print(f"Train Size: {len(train_dataset)}")
print(f"Val Size: {len(val_dataset)}")
print(f"Test Size: {len(test_dataset)}")

In [ ]:
# Weighted Random Sampler

train_targets = targets[train_idx]
class_sample_count = np.array([len(np.where(train_targets == t)[0]) for t in np.unique(train_targets)])
weight = 1. / class_sample_count
samples_weight = np.array([weight[t] for t in train_targets])
samples_weight = torch.from_numpy(samples_weight)

sampler = WeightedRandomSampler(
    weights=samples_weight,
    num_samples=len(samples_weight),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))
oreo_count = (labels == full_dataset.class_to_idx['oreo']).sum().item()
not_oreo_count = (labels == full_dataset.class_to_idx['not_oreo']).sum().item()

print(f"Batch size: {len(labels)}")
print(f"Oreo images: {oreo_count}")
print(f"Not Oreo images: {not_oreo_count}")

In [ ]:
# logistic Regression
class LogRegression:
    def __init__(self, lr, epochs, max_iter, batch_size=32):
        self.lr = lr
        self.epochs = epochs
        self.max_iter = max_iter
        self.batch_size = batch_size
        self.weights = None

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def loss(self, y, y_pred):
        N = len(y)
        y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10)
        return - (1 / N) * np.sum(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))

    def store(self, x, y):
        self.weights = np.zeros(x.shape[1])
        prev_loss = float('inf')
        N = len(y)

        for i in range(self.epochs):
            indices = np.random.permutation(N)
            xs = x[indices]
            ys = y[indices]

            for j in range(0, N, self.batch_size):
                x_batch = xs[j : j + self.batch_size]
                y_batch = ys[j : j + self.batch_size]

                z = np.dot(x_batch, self.weights)

                y_pred = self.sigmoid(z)

                gradient = (1 / len(y_batch)) * np.dot(x_batch.T, y_batch - y_pred)
                self.weights -= self.lr * gradient

            full_pred = self.sigmoid(np.dot(x, self.weights))
            current_loss = self.loss(y, full_pred)

            if abs(current_loss - prev_loss) < self.max_iter:
                break
            prev_loss = current_loss

    def predict(self, x):
        y_pred = self.sigmoid(np.dot(x, self.weights))
        return (y_pred >= 0.5).astype(int)

In [ ]:
# logistic validation

In [ ]:
# K-NN
class KNN:
    def __init__(self, k):
        self.k = k

    # Stores training data and labels for prediction
    def store(self, x, y):
        self.x_train = x
        self.y_train = y

    # Use euclidean distance to find nearest K neighbors and predict if Oreo or not
    def predict(self, x):
        test_squared = np.sum(x**2, axis=1, keepdims=True)
        train_squared = np.sum(self.x_train**2, axis=1)

        distances = np.sqrt(np.maximum(test_squared + train_squared - 2 * np.dot(x, self.x_train.T), 0))

        # Finds majority label among K nearest neighbors
        predictions = []
        for i in range(distances.shape[0]):
            knn_indices = np.argsort(distances[i])[:self.k]
            knn_labels = self.y_train[knn_indices]
            predicted_label = np.bincount(knn_labels).argmax()
            predictions.append(predicted_label)

        return np.array(predictions)
    
knn = KNN(5)
knn.store(x_train, y_train)

predictions = knn.predict(x_test)

print(f"Predictions: {predictions}")
print(f"Actual: {y_test}")

In [ ]:
# K-NN validation


In [ ]:
# CNN

In [ ]:
# CNN validation

In [ ]:
# evaluation

In [ ]:
# run things